# Create Embedding using Word2Vec

### Task:
You are given corpus:
```
i love machine learning.
machine learning is fun.
deep learning is powerful and fun.
i enjoy learning new things.
artificial intelligence is the future and I love learning it.
```
Create a embeddings of above using word2vec 

## Install gensim library

In [1]:
!pip install gensim==4.4.0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import gensim

print(gensim.__version__) # 4.4.0

4.4.0


In [2]:
from gensim.models import Word2Vec

## 1) Create Sample Corpus (Tokenized Text)

 Gensim expects **list of list of words**


In [3]:
# Define the raw text lines
text = """
i love machine learning.
machine learning is fun.
deep learning is powerful and fun.
i enjoy learning new things.
artificial intelligence is the future and I love learning it.
"""

# Split the text by lines, clean commas/spaces, and split each line into words
corpus = [line.strip().replace(".", "").split() for line in text.strip().split("\n")]
corpus

[['i', 'love', 'machine', 'learning'],
 ['machine', 'learning', 'is', 'fun'],
 ['deep', 'learning', 'is', 'powerful', 'and', 'fun'],
 ['i', 'enjoy', 'learning', 'new', 'things'],
 ['artificial',
  'intelligence',
  'is',
  'the',
  'future',
  'and',
  'I',
  'love',
  'learning',
  'it']]

## 2) Train Word2Vec Model

In [5]:
# Convert words to vector

w2v_model = Word2Vec(
    sentences=corpus,
    
    vector_size=20, # Size of word vectors: dimensionality of the word embeddings.
    
    window=3,       # Context window: A window of 3 means the model looks at 3 words
                    # before and 3 words after the target word to learn its context.
    
    min_count=1,    # Include all words: This sets the minimum frequency threshold for a word to be included in the model's vocabulary. 
                    # In large datasets, this is usually set to 5+ to ignore rare words and typos
    
    workers=2      #  use 2 CPU cores to train the model in parallel, making the training process faster.
)


## 3) Word Embeddings (Core Idea)
- Find embeddings for a word 'learning'

In [6]:
vector = w2v_model.wv['learning']

print(vector)

[-0.00268114  0.00118216  0.02551675  0.04504637 -0.04651475 -0.03558404
  0.03229436  0.04486494 -0.02507714 -0.01881686  0.03690252 -0.00766736
 -0.02268307  0.03277026 -0.0243008  -0.00908009  0.0143829   0.00495937
 -0.04142607 -0.04724409]


##  4) Vocabulary
- Lets view all the vocabulary

In [7]:
print(w2v_model.wv.index_to_key)

['learning', 'is', 'and', 'fun', 'machine', 'love', 'i', 'it', 'I', 'future', 'the', 'intelligence', 'artificial', 'things', 'new', 'enjoy', 'powerful', 'deep']


## 5) Similar Words (VERY IMPORTANT DEMO)
- Find similarity score between word **'learning'** and other words

In [8]:
w2v_model.wv.most_similar('learning')

[('love', 0.39641639590263367),
 ('powerful', 0.3511616289615631),
 ('things', 0.3365645408630371),
 ('artificial', 0.22743260860443115),
 ('intelligence', 0.14539529383182526),
 ('fun', 0.055946141481399536),
 ('enjoy', 0.05372633785009384),
 ('is', 0.05014949291944504),
 ('deep', -0.014511939138174057),
 ('it', -0.030630286782979965)]

##  Similarity Score Between 2 Words

In [10]:
sim = w2v_model.wv.similarity('artificial', 'learning')
print(sim)

0.2274326


## 6) Check if Words are in Vocabulary

In [16]:
print("learning" in w2v_model.wv)

True


In [9]:
print("ai" in w2v_model.wv)

False


## 7) Save & Load Model

In [11]:
w2v_model.save("word2vec.model")

In [12]:
# Load later
model_loaded = Word2Vec.load("word2vec.model")

In [13]:
model_loaded.wv.most_similar('learning')

[('love', 0.39641639590263367),
 ('powerful', 0.3511616289615631),
 ('things', 0.3365645408630371),
 ('artificial', 0.22743260860443115),
 ('intelligence', 0.14539529383182526),
 ('fun', 0.055946141481399536),
 ('enjoy', 0.05372633785009384),
 ('is', 0.05014949291944504),
 ('deep', -0.014511939138174057),
 ('it', -0.030630286782979965)]

# OPTIONAL

## 8) Application: Feed the vectors to RNN
- Convert sentence -> words -> vectors -> feed into LSTM

In [14]:
sentence = ["i", "love", "learning"]

vectors = [w2v_model.wv[word] for word in sentence]

print(vectors)


[array([ 0.01174794, -0.02260485,  0.0419188 , -0.04929366,  0.03383001,
        0.01458636, -0.02465954,  0.02199213, -0.00870167,  0.0335678 ,
        0.0498282 , -0.02182942, -0.00299059, -0.02847647,  0.01925817,
        0.01393841,  0.03447564,  0.0305202 ,  0.0476854 ,  0.0463738 ],
      dtype=float32), array([-0.04309844,  0.01832869,  0.02594942,  0.02870969,  0.03733459,
       -0.03083838,  0.00552807,  0.03023641, -0.01420025, -0.03086761,
       -0.00205112, -0.04184474, -0.02800006,  0.03552269,  0.0167627 ,
        0.03612835,  0.03400124,  0.03765371, -0.01894577, -0.00280903],
      dtype=float32), array([-0.00268114,  0.00118216,  0.02551675,  0.04504637, -0.04651475,
       -0.03558404,  0.03229436,  0.04486494, -0.02507714, -0.01881686,
        0.03690252, -0.00766736, -0.02268307,  0.03277026, -0.0243008 ,
       -0.00908009,  0.0143829 ,  0.00495937, -0.04142607, -0.04724409],
      dtype=float32)]


Now this becomes input to LSTM

### Pipeline:

```
Text → Tokenization → Word2Vec → Vectors → LSTM
```
